# Basic classification algorithms using the sklearn library

Train and classify the data. Perform the following steps:
1. Load the data from the website, read it, and print the column names and dataset size
2. Handle missing values (fill them if possible or remove them)
3. Visualize the data: build a heatmap showing correlations
   between features and with the target variable (labels); build histograms
   of feature distributions and boxplot diagrams of features relative to
   the target variable (if there are too many features, limit it to a few).
4. Scale the data
5. Train the following classifiers:
    kNN (sklearn.neighbors.KNeighborsClassifier)
    train a decision tree, visualize it (using
    sklearn.tree.export_graphviz and pydot)
    SVM (sklearn.svm.SVC)

6. Train ensemble classifiers (Random Forest, AdaBoost, Gradient Boost)
   Tune the optimal parameters for each model:
    - Number of nearest neighbors for kNN
    - For SVM, consider linear and rbf kernels; using grid search
      (sklearn.model_selection.GridSearchCV), find the optimal C and gamma
    - For ensemble methods, find the optimal parameter values using
    grid search

Among the selected optimal models of each class, choose the best one
using f1-score as the main criterion
Display sklearn.metrics.classification_report and sklearn.metrics.confusion_matrix

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import export_graphviz
import pydotplus

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

### Load the data from the website, read it, and print the column names and dataset size

In [ ]:
df = pd.read_csv("bird.csv")
print(df.head())

In [ ]:
print("Dataset columns:", df.columns)
print("Dataset shape:", df.shape)

### Handle missing values (fill them if possible or remove them)

In [ ]:
df.isna().sum()

In [ ]:
df = df.fillna(df.mean(numeric_only=True))
df = df.drop(columns=["id"])

In [ ]:
df.isna().sum()

### Data visualization (Correlation heatmap)

In [ ]:
feature_columns = [
    "huml",
    "humw",
    "ulnal",
    "ulnaw",
    "feml",
    "femw",
    "tibl",
    "tibw",
    "tarl",
    "tarw",
]
corr = df[feature_columns].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    cmap=sns.light_palette("#00304e", as_cmap=True),
    square=True,
    cbar=True,
    ax=ax,
    annot=True,
    annot_kws={"fontsize": 8},
    fmt=".2f",
)
ax.set_title("Correlation Matrix of Bird Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
plt.close()

### Distribution histograms for each feature

In [ ]:
for bird_type in feature_columns:
    sns.displot(df, x=bird_type)

### Boxplots of features by bird type

In [ ]:
for bird_type in feature_columns:
    fig = plt.figure()
    ax = sns.boxplot(x="type", y=bird_type, data=df)

### Prepare data for training

In [ ]:
# Split features and target variable
X = df.drop("type", axis=1).values
y = df["type"].values.reshape(-1, 1)

# Split dataset into training and test sets (70-30 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

### Model training and evaluation function

In [ ]:
def run_ml_model(ml_model, parameters, X_train, y_train, model_name):
    """
    Train and evaluate a machine learning model with hyperparameter tuning.

    Parameters:
    -----------
    ml_model : estimator object
        The machine learning model to train
    parameters : dict
        Grid search parameters
    X_train : array-like
        Training features
    y_train : array-like
        Training labels
    model_name : str
        Name of the model for saving figures

    Returns:
    --------
    ml_model : GridSearchCV object
        Trained model with best parameters
    """
    # Create pipeline with scaling and model
    steps = [
        ("scaler", StandardScaler()),  # Feature scaling
        ("model", ml_model),
    ]
    model_pipe = Pipeline(steps)

    # Perform grid search with cross-validation
    print(f"\n{'=' * 60}")
    print(f"Training {model_name}...")
    print(f"{'=' * 60}")

    ml_model = GridSearchCV(model_pipe, parameters, cv=3, n_jobs=-1, verbose=1)
    ml_model = ml_model.fit(X_train, y_train.ravel())

    # Predictions
    y_pred_train = ml_model.predict(X_train)
    y_pred_test = ml_model.predict(X_test)

    # Calculate accuracies
    accuracy_train = accuracy_score(y_train, y_pred_train)
    accuracy_test = accuracy_score(y_test, y_pred_test)

    # Display results
    print(f"\nTraining set accuracy: {accuracy_train:.4f}")
    print(f"Test set accuracy: {accuracy_test:.4f}")
    print(f"\nBest parameters: {ml_model.best_params_}")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_test))

    # Save confusion matrix
    fig, ax = plt.subplots(figsize=(10, 8))
    cm = confusion_matrix(y_test, y_pred_test)
    sns.heatmap(
        cm, annot=True, cmap="viridis", fmt=".0f", ax=ax, cbar_kws={"label": "Count"}
    )
    ax.set_title(f"Confusion Matrix - {model_name}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close()

    return ml_model

### Train k-Nearest Neighbors

In [ ]:
parameters_knn = {
    "model__algorithm": ["brute"],
    "model__leaf_size": [30, 50, 70, 90, 110],
    "model__metric": ["minkowski"],
    "model__p": [2],
    "model__n_neighbors": [3, 5, 11, 19],
    "model__weights": ["uniform", "distance"],
    "model__n_jobs": [-1],
}

model_knn = run_ml_model(
    KNeighborsClassifier(), parameters_knn, X_train, y_train, "k-Nearest Neighbors"
)

### Train Random Forest

In [ ]:
parameters_rfc = {
    "model__n_estimators": [100, 200],
    "model__max_features": ["sqrt", "log2"],
    "model__max_depth": [10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__bootstrap": [True, False],
    "model__criterion": ["gini"],
    "model__random_state": [42],
}
model_rfc = run_ml_model(
    RandomForestClassifier(), parameters_rfc, X_train, y_train, "Random Forest"
)

### Train AdaBoost

In [ ]:
parameters_ab = {
    "model__learning_rate": [0.5, 1, 2, 3],
    "model__n_estimators": [50, 100, 200, 300, 400, 600],
    "model__random_state": [42, 56],
}

model_ab = run_ml_model(
    AdaBoostClassifier(), parameters_ab, X_train, y_train, "AdaBoost"
)

### Train Support Vector Machine

In [ ]:
parameters_svc = {
    "model__kernel": ["linear", "rbf"],
    "model__C": [1, 10, 100, 1000, 10000],
    "model__random_state": [42],
    "model__gamma": ["scale", "auto"],
}
model_svc = run_ml_model(SVC(), parameters_svc, X_train, y_train, "SVM")

### Train Gradient Boosting

In [ ]:
parameters_gb = {
    "model__n_estimators": [100, 200],
    "model__loss": ["log_loss"],
    "model__learning_rate": [0.01, 0.001, 0.1, 0.2],
    "model__max_features": ["log2", "sqrt"],
    "model__criterion": ["friedman_mse", "squared_error"],
    "model__random_state": [42],
}

model_gb = run_ml_model(
    GradientBoostingClassifier(), parameters_gb, X_train, y_train, "Gradient Boosting"
)

### Train Decision Tree

In [ ]:
parameters_dt = {
    "model__max_depth": np.arange(1, 10),
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__min_samples_split": np.arange(2, 11),
    "model__criterion": ["gini", "entropy"],
    "model__random_state": [42],
}
model_dt = run_ml_model(
    DecisionTreeClassifier(), parameters_dt, X_train, y_train, "Decision Tree"
)

### Building tree for Decision Tree Classifier

In [ ]:
classifier_dt = DecisionTreeClassifier(
    criterion="gini",
    max_depth=9,
    min_samples_leaf=1,
    min_samples_split=4,
    random_state=42,
).fit(X_train, y_train.ravel())

In [ ]:
dot_data = export_graphviz(classifier_dt, filled=True, rounded=True)
graph = pydotplus.graph_from_dot_data(dot_data)
graph.write_png("decision_tree.png")